In [ ]:
# I run this on google colab due to conflict between module version in the requirements.txt

In [1]:
!pip install -qU langchain langchain-pinecone langchain-openai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.24 requires simsimd>=5.9.2, but you have simsimd 3.9.0 which is incompatible.


In [2]:
!pip install --upgrade langchain-pinecone

  Using cached langchain_pinecone-0.2.12-py3-none-any.whl.metadata (8.6 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached simsimd-6.5.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (70 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.0.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
Using cached langchain_pinecone-0.2.12-py3-none-any.whl (25 kB)
Using cached langchain_core-0.3.79-py3-none-any.whl (449 kB)
Using cached langchain_openai-0.3.35-py3-none-any.whl (75 kB)
Using cached simsimd-6.5.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (1.1 MB)
  Attempting uninstall: simsimd
    Found existing installation: simsimd 3.9.0
    Uninstalling simsimd-3.9.0:
      Successfully uninstalled simsimd-3.9.0
  Attem

In [4]:
from pinecone import Pinecone

API_KEY = "PUT_YOUR_API_KEY_HERE"
pc = Pinecone(api_key=API_KEY)

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1024, api_key="UNK")

In [6]:
from pinecone import ServerlessSpec

index_name = "rag-system"

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
  pc.create_index(
      name=index_name,
      dimension=1024,
      metric="cosine",
      spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )

index = pc.Index(index_name)

In [7]:
index

In [9]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [10]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [11]:
vector_store.add_documents(documents)

['af0ad798-fc5e-4922-bcb5-010f992b0ef2',
 '18ae5453-9c40-4762-b04c-f2976026045f',
 '1e8f436f-f9b4-42d2-93e9-0a830c18808e',
 'bf098ae5-0871-4297-902f-f81cda0bb636',
 '3dde9a96-c885-4c9c-88c9-56efd5de16fc',
 'ec57f04c-b2d5-4369-b82a-8272f4f7e3e7',
 '50252e3b-a13f-47bf-9b9d-0631c467bcd9',
 '3fd2a00f-3e97-4caf-82b1-b6238f95dcc7',
 'a1d7095e-0a6d-4065-9abd-d5f64e1e4d0f',
 '21d4b114-d10c-48ab-b0aa-bda82cd70b60']

In [12]:
results = vector_store.similarity_search(
    "How about the stock market",
    k=3
)

for res in results:
  print(f"Metadat: {res.metadata}")
  print(f"Page Content: {res.page_content}")

Metadat: {'source': 'news'}
Page Content: The stock market is down 500 points today due to fears of a recession.
Metadat: {'source': 'news'}
Page Content: The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.
Metadat: {'source': 'tweet'}
Page Content: Building an exciting new project with LangChain - come check it out!
